In [ ]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm
from pathlib import Path

BASE_DIR = Path("D:/Projects/Masters/Data")
DATASET_ROOT     = BASE_DIR / "AR_dataset"
BEE_DATASET_ROOT = BASE_DIR / "DET_data_sliced_split"
OUTPUT_DIR       = BASE_DIR / "AR_merged_dataset"
CONTEXT_SIZE     = 320                 # adjust this if needed

BACKGROUND_RATIO = 0.25  # background will be 25% of total dataset

In [7]:
ACTION_CLASSES = sorted([
    f for f in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, f))
])

print("Detected action classes:")
for idx, action in enumerate(ACTION_CLASSES):
    print(f"  {idx} → {action}")

Detected action classes:
  0 → fanning
  1 → trophallaxis


In [8]:
def audit_split(action_folder, split):
    images_path = os.path.join(DATASET_ROOT, action_folder, split, "images")
    labels_path = os.path.join(DATASET_ROOT, action_folder, split, "labels")
    
    annotated = []
    unannotated = []
    
    for img_file in os.listdir(images_path):
        stem = os.path.splitext(img_file)[0]
        label_file = os.path.join(labels_path, stem + ".txt")
        
        is_annotated = False
        if os.path.exists(label_file):
            with open(label_file, "r") as f:
                line = f.readline().strip().split()
            if len(line) >= 5:
                is_annotated = True
        
        if is_annotated:
            annotated.append(img_file)
        else:
            unannotated.append(img_file)
    
    return annotated, unannotated

def audit_all_splits():
    summary = {}
    for action in tqdm(ACTION_CLASSES, desc="Auditing dataset"):
        summary[action] = {}
        for split in ["train", "val"]:
            annotated, unannotated = audit_split(action, split)
            summary[action][split] = {
                "annotated": annotated,
                "unannotated": unannotated
            }
    return summary

def crop_and_save(img_path, label_path, output_path, context_size=CONTEXT_SIZE):
    frame = cv2.imread(img_path)
    if frame is None:
        return False
    
    h, w = frame.shape[:2]
    
    with open(label_path, "r") as f:
        line = f.readline().strip().split()
    
    if len(line) < 5:
        return False
    
    _, cx, cy, bw, bh = map(float, line[:5])
    
    center_x = int(cx * w)
    center_y = int(cy * h)
    half = context_size // 2
    
    x1 = max(0, center_x - half)
    y1 = max(0, center_y - half)
    x2 = min(w, center_x + half)
    y2 = min(h, center_y + half)
    
    crop = frame[y1:y2, x1:x2]
    
    if crop.size == 0:
        return False
    
    cv2.imwrite(output_path, crop)
    return True

def convert_dataset(summary):
    stats = {action: {"train": 0, "val": 0} for action in ACTION_CLASSES}
    failed = []

    total = sum(
        len(summary[action][split]["annotated"])
        for action in ACTION_CLASSES
        for split in ["train", "val"]
    )

    with tqdm(total=total, desc="Converting dataset", unit="img") as pbar:
        for action in ACTION_CLASSES:
            for split in ["train", "val"]:
                for img_file in summary[action][split]["annotated"]:
                    stem = os.path.splitext(img_file)[0]
                    img_path    = os.path.join(DATASET_ROOT, action, split, "images", img_file)
                    label_path  = os.path.join(DATASET_ROOT, action, split, "labels", stem + ".txt")
                    output_path = os.path.join(OUTPUT_DIR, split, action, f"{action}_{img_file}")

                    success = crop_and_save(img_path, label_path, output_path)
                    if success:
                        stats[action][split] += 1
                    else:
                        failed.append(img_path)

                    pbar.update(1)
                    pbar.set_postfix({action: stats[action][split]})

    return stats, failed

def generate_background_samples(split, limit, context_size=CONTEXT_SIZE):
    images_path = os.path.join(BEE_DATASET_ROOT, split, "images")
    labels_path = os.path.join(BEE_DATASET_ROOT, split, "labels")
    output_path = os.path.join(OUTPUT_DIR, split, "background")

    all_images = [f for f in os.listdir(images_path) if f.endswith((".jpg", ".png"))]
    random.shuffle(all_images)

    saved = 0
    failed = 0

    for img_file in tqdm(all_images, desc=f"Background {split}"):
        if saved >= limit:
            break

        stem = os.path.splitext(img_file)[0]
        label_file = os.path.join(labels_path, stem + ".txt")

        if not os.path.exists(label_file):
            continue

        frame = cv2.imread(os.path.join(images_path, img_file))
        if frame is None:
            continue

        h, w = frame.shape[:2]

        with open(label_file, "r") as f:
            lines = f.readlines()

        for line in lines:
            if saved >= limit:
                break

            parts = line.strip().split()
            if len(parts) < 5:
                continue

            _, cx, cy, bw, bh = map(float, parts[:5])

            center_x = int(cx * w)
            center_y = int(cy * h)
            half = context_size // 2

            x1 = max(0, center_x - half)
            y1 = max(0, center_y - half)
            x2 = min(w, center_x + half)
            y2 = min(h, center_y + half)

            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                failed += 1
                continue

            out_name = f"background_{stem}_{saved}.jpg"
            cv2.imwrite(os.path.join(output_path, out_name), crop)
            saved += 1

    print(f"{split} → saved: {saved} | failed: {failed}")
    return saved

In [9]:
for split in ["train", "val"]:
    for action in ACTION_CLASSES:
        os.makedirs(os.path.join(OUTPUT_DIR, split, action), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "background"), exist_ok=True)

print("Created folder structure:")
for split in ["train", "val"]:
    print(f"\n  {split}/")
    for folder in os.listdir(os.path.join(OUTPUT_DIR, split)):
        print(f"    {folder}/")

Created folder structure:

  train/
    background/
    fanning/
    trophallaxis/

  val/
    background/
    fanning/
    trophallaxis/


In [10]:
# Step 1: Audit action dataset
summary = audit_all_splits()

total_train = 0
total_val = 0

for action, splits in summary.items():
    print(f"\n{action}")
    for split, data in splits.items():
        count = len(data['annotated'])
        print(f"  {split} → {count} annotated, {len(data['unannotated'])} unannotated")
        if split == "train":
            total_train += count
        else:
            total_val += count
bg_multiplier = BACKGROUND_RATIO / (1 - BACKGROUND_RATIO)
BACKGROUND_TRAIN_LIMIT = int(total_train * bg_multiplier)
BACKGROUND_VAL_LIMIT   = int(total_val * bg_multiplier)

print(f"\nTotal action train samples: {total_train}")
print(f"Total action val samples:   {total_val}")
print(f"\nBackground train limit: {BACKGROUND_TRAIN_LIMIT} ({BACKGROUND_RATIO*100:.0f}% of total)")
print(f"Background val limit:   {BACKGROUND_VAL_LIMIT} ({BACKGROUND_RATIO*100:.0f}% of total)")

Auditing dataset: 100%|██████████| 2/2 [04:39<00:00, 139.65s/it]


fanning
  train → 12199 annotated, 2201 unannotated
  val → 2904 annotated, 696 unannotated

trophallaxis
  train → 13506 annotated, 2372 unannotated
  val → 3324 annotated, 1278 unannotated

Total action train samples: 25705
Total action val samples:   6228

Background train limit: 8568 (25% of total)
Background val limit:   2076 (25% of total)


In [ ]:
# Step 2: Convert action crops
stats, failed = convert_dataset(summary)

print("\nConversion complete!")
for action, splits in stats.items():
    print(f"  {action}")
    for split, count in splits.items():
        print(f"    {split} → {count} images")
print(f"\nFailed: {len(failed)}")

In [ ]:
# Step 3: Generate background samples

generate_background_samples("train", BACKGROUND_TRAIN_LIMIT)
generate_background_samples("val",   BACKGROUND_VAL_LIMIT)

Background train:   4%|▍         | 897/21000 [00:22<08:29, 39.46it/s]


train → saved: 3000 | failed: 0


Background val:   7%|▋         | 313/4500 [00:08<01:50, 37.82it/s]

val → saved: 1000 | failed: 0


1000

In [ ]:
# Visually verify crops from each class
for folder in os.listdir(os.path.join(OUTPUT_DIR, "train")):
    folder_path = os.path.join(OUTPUT_DIR, "train", folder)
    samples = os.listdir(folder_path)[:2]
    for s in samples:
        img = cv2.imread(os.path.join(folder_path, s))
        if img is not None:
            cv2.imshow(f"{folder} - {s}", img)

cv2.waitKey(0)
cv2.destroyAllWindows()